In [ ]:
# import necessary packages
import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# define colors used in this work
paton_hot_pink = '#d00d44'
rosewood_plum = '#9B6B8E'     
rosewood_sage = '#7BA882'     
rosewood_gold = '#D4A574'       
rosewood_slate = '#5B7FA6'     
rosewood_rose = '#C9858C'        
rosewood_charcoal = '#4A4A4A'    
rosewood_cream = '#F5EDE0' 
tol_grey = '#BBBBBB'

# define plotting parameters
sns.set(context='talk', style='ticks',
        color_codes=True, rc={'legend.frameon': False})
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
plt.rcParams["font.family"] = 'Avenir'
plt.rcParams.update({'font.size': 24})
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# read the data
full_dataset = 'hammett_predictions.csv'

In [ ]:
# drop the predicted_hammett column in the dataframe; we will predict them via the same method in this notebook
df_all = pd.read_csv(full_dataset)
df_all = df_all.drop(columns=['predicted_hammett'])

In [ ]:
# separate the meta and para substituents into separate dataframes
meta = df_all['nitrile_id'].str.contains('meta')
para = df_all['nitrile_id'].str.contains('para')
df_meta = df_all[meta]
df_para = df_all[para]

# remove H from the para substituents so it is not double-counted in the combined dataset
df_all = df_all[df_all['nitrile_id'] != 'para_nitrile_032']

# perform a linear regression to find the relationship between the calculated CM5 charge and the experimental Hammett parameter
x = df_all[['cm5_charge']]
y = df_all['hammett_parameter']

regr = linear_model.LinearRegression()
regr.fit(x, y)
r_value = r2_score(y, regr.predict(x))

slope = regr.coef_[0]
intercept = regr.intercept_
print(f"Equation of the line: y = {slope:.3f}x + {intercept:.2f}")

# 95% confidence interval on the slope (and intercept) using statsmodels ---
X_sm = sm.add_constant(x)  # adds intercept column
model = sm.OLS(y, X_sm).fit()

conf_int = model.conf_int(alpha=0.05)  # 95% CI
slope_ci_low, slope_ci_high = conf_int.loc['cm5_charge']
intercept_ci_low, intercept_ci_high = conf_int.loc['const']

slope_se = model.bse['cm5_charge']
intercept_se = model.bse['const']

print(f"Slope: {slope:.4f}  (SE = {slope_se:.4f})")
print(f"Intercept: {intercept:.4f}  (SE = {intercept_se:.4f})")
print(f"Slope 95% CI: [{slope_ci_low:.4f}, {slope_ci_high:.4f}]")
print(f"Intercept 95% CI: [{intercept_ci_low:.4f}, {intercept_ci_high:.4f}]")

#OPTIONAL – visualize the data and best fit line
plt.scatter(x, y, color=rosewood_sage, alpha=0.3, edgecolors=rosewood_sage, s=35)  # Use a specific color from the colors dictionary
plt.xlabel('Calculated CM5 Charge')
plt.ylabel('Experimental Hammett Parameter')
plt.xlim(-0.37, -0.31)
plt.ylim(-0.75, 1.5)
plt.plot(x, regr.predict(x), color='black', linewidth=1, label='Best Fit Line')
plt.xticks(np.linspace(-0.36, -0.32, 4), [f"{tick:.2f}" for tick in np.linspace(-0.36, -0.32, 4)])
plt.yticks(np.linspace(-0.5, 1.25, 4), [f"{tick:.2f}" for tick in np.linspace(-0.5, 1.25, 4)])
plt.annotate(f'y = {slope:.2f}x + {intercept:.2f}', xy=(0.1,0.80), xycoords='axes fraction', fontsize=20, fontweight='bold')

# use the regression equation to predict Hammett constants
df_all['predicted_hammett'] = (slope * df_all['cm5_charge'] + intercept)
df_meta['predicted_hammett'] = (slope * df_meta['cm5_charge'] + intercept)
df_para['predicted_hammett'] = (slope * df_para['cm5_charge'] + intercept)

plt.show()

In [ ]:
# visualize the predicted vs. experimental Hammett parameters
x = df_meta['hammett_parameter'].values.reshape(-1, 1)
y = df_meta['predicted_hammett'].values.reshape(-1, 1)
z = df_para['hammett_parameter'].values.reshape(-1, 1)
c = df_para['predicted_hammett'].values.reshape(-1, 1)
a = df_all['hammett_parameter'].values.reshape(-1,1)
b = df_all['predicted_hammett'].values.reshape(-1,1)

# perform a linear regression to determine R^2 and MAE
regr_meta = linear_model.LinearRegression()
regr_meta.fit(x, y)
r_value_meta = r2_score(y, regr_meta.predict(x))

mae_meta = mean_absolute_error(y, x)

regr_para = linear_model.LinearRegression()
regr_para.fit(z, c)
r_value_para = r2_score(c, regr_para.predict(z))

mae_para = mean_absolute_error(c, z)

regr_all = linear_model.LinearRegression()
regr_all.fit(a, b)
r_value_all = r2_score(b, regr_all.predict(a))

mae_all = mean_absolute_error(b, a)

# Visualize the results in a three-panel scatter plot
fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize=(10, 5), sharex=True, sharey=True)
ax1.scatter(x, y, color=rosewood_rose, alpha=0.30, edgecolors=rosewood_rose, s=35)
ax2.scatter(z, c, color=rosewood_sage, alpha=0.30, edgecolors=rosewood_sage, s=35)
ax3.scatter(a, b, color=rosewood_slate, alpha=0.20, edgecolors=rosewood_slate, s=35)
ax1.set_xlim(-0.9, 1.5)
ax2.set_xlim(-0.9, 1.5)
ax3.set_xlim(-0.9, 1.5)
ax1.set_xticks(np.linspace(-0.5, 1.0, 4))
ax2.set_xticks(np.linspace(-0.5, 1.0, 4))
ax3.set_xticks(np.linspace(-0.5, 1.0, 4))
ax1.set_ylim(-0.75, 1.5)
ax2.set_ylim(-0.75, 1.5)
ax3.set_ylim(-0.75, 1.5)
ax1.set_yticks(np.linspace(-0.5, 1.0, 4))
ax2.set_yticks(np.linspace(-0.5, 1.0, 4))
ax3.set_yticks(np.linspace(-0.5, 1.0, 4))
ax1.axline((0, 0), slope=1, color=tol_grey, linestyle="--")
ax2.axline((0, 0), slope=1, color=tol_grey, linestyle="--")
ax3.axline((0, 0), slope=1, color=tol_grey, linestyle="--")
ax1.annotate(f'R$^2$ = {"{:.2f}".format(r_value_meta)}', xy=(0.045,0.90), xycoords='axes fraction', fontsize=16)
ax1.annotate(f'MAE = {"{:.2f}".format(mae_meta)}', xy=(0.045,0.83), xycoords='axes fraction', fontsize=16)
ax1.annotate(f'n = {len(df_meta)}', xy=(0.045,0.76), xycoords='axes fraction', fontsize=16)
ax1.annotate('$\sigma$$_m$', xy=(0.7,0.1), xycoords='axes fraction', fontsize=18)
ax2.annotate(f'R$^2$ = {"{:.2f}".format(r_value_para)}', xy=(0.045,0.90), xycoords='axes fraction', fontsize=16)
ax2.annotate(f'MAE = {"{:.2f}".format(mae_para)}', xy=(0.045,0.83), xycoords='axes fraction', fontsize=16)
ax2.annotate(f'n = {len(df_para)}', xy=(0.045,0.76), xycoords='axes fraction', fontsize=16)
ax2.annotate('$\sigma$$_p$', xy=(0.7,0.1), xycoords='axes fraction', fontsize=18)
ax3.annotate(f'R$^2$ = {"{:.2f}".format(r_value_all)}', xy=(0.045,0.90), xycoords='axes fraction', fontsize=16)
ax3.annotate(f'MAE = {"{:.2f}".format(mae_all)}', xy=(0.045,0.83), xycoords='axes fraction', fontsize=16)
ax3.annotate(f'n = {len(df_all)}', xy=(0.045,0.76), xycoords='axes fraction', fontsize=16)
ax3.annotate('$\sigma$$_m$ & $\sigma$$_p$', xy=(0.55,0.1), xycoords='axes fraction', fontsize=18)
fig.text(0.5, 0.01, 'Experimental Hammett Parameter', ha='center', va='center', fontsize=16)
fig.text(0.04, 0.5, 'Predicted Hammett Parameter', ha='center', va='center', rotation='vertical', fontsize=16)

# plt.savefig('model-results-three-panel.svg', dpi=400, bbox_inches='tight')

plt.show()

In [ ]:
# 95% CI on slope for each regression
X_meta = sm.add_constant(x)
model_meta = sm.OLS(y, X_meta).fit()
slope_meta_ci = model_meta.conf_int(alpha=0.05)[1]  # row 1 = slope

X_para = sm.add_constant(z)
model_para = sm.OLS(c, X_para).fit()
slope_para_ci = model_para.conf_int(alpha=0.05)[1]

X_all = sm.add_constant(a)
model_all = sm.OLS(b, X_all).fit()
slope_all_ci = model_all.conf_int(alpha=0.05)[1]

print(f'slope para: {regr_para.coef_[0][0]:.2f}  (95% CI: [{slope_para_ci[0]:.2f}, {slope_para_ci[1]:.2f}])')
print(f'slope meta: {regr_meta.coef_[0][0]:.2f}  (95% CI: [{slope_meta_ci[0]:.2f}, {slope_meta_ci[1]:.2f}])')
print(f'slope all: {regr_all.coef_[0][0]:.2f}  (95% CI: [{slope_all_ci[0]:.2f}, {slope_all_ci[1]:.2f}])')